In [10]:
#!/usr/bin/env python
# coding: utf-8

import os
import subprocess
import concurrent.futures
from joblib import Parallel, delayed

In [11]:
    
for num_nodes in  [32]:
    
    # Fetch all existing instance names matching tsm-sc-*
    fetch_cmd = f'''
    gcloud compute instances list \
        --filter="name~'tsm-sc-'" \
        --format="value(name)"
    '''
    instances_to_delete = subprocess.check_output(fetch_cmd, shell=True).decode().strip().split('\n')
    instances_to_delete = [name for name in instances_to_delete if name]  # Remove empty entries
    
    print("\n➡ Existing instances to delete:", instances_to_delete)
    
    if instances_to_delete:
        # Use parallel deletion
        def delete_instance(instance_name):
            cmd = f'''
            gcloud compute instances delete {instance_name} \
                --zone={zone} \
                --project={project} \
                --quiet
            '''
            print(f"Deleting: {instance_name}")
            return subprocess.call(cmd, shell=True)
    
        with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
            futures = [executor.submit(delete_instance, inst) for inst in instances_to_delete]
            concurrent.futures.wait(futures)
    
        print("🧹 All old tsm-sc-* instances deleted.\n")
    else:
        print("✔ No previous instances found to delete.\n")
    
    


➡ Existing instances to delete: []
✔ No previous instances found to delete.



In [12]:
    
    project = "ucr-ursa-major-lesani-lab"
    zone = "us-central1-c"
    machine_type = "e2-highmem-2"
    image_family = "tsm-sc-family"  # your custom image
    subnet = "default"
    gcp_username = "tejas"
    

    # Cleanup any existing instances with same prefix
    os.system(f'gcloud compute instances delete --zone={zone} --quiet '
              f'$(gcloud compute instances list --filter="name~\'tsm-sc-\'" --format="value(name)")')
    
    # Create commands list
    commands = []
    
    for i in range(num_nodes):
        cmd = f'''
        gcloud compute instances create tsm-sc-{i:03} \
            --project={project} \
            --zone={zone} \
            --machine-type={machine_type} \
            --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
            --can-ip-forward \
            --maintenance-policy=MIGRATE \
            --provisioning-model=STANDARD \
            --service-account=961693926925-compute@developer.gserviceaccount.com \
            --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
            --tags=http-server,https-server \
            --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
            --no-shielded-secure-boot \
            --shielded-vtpm \
            --shielded-integrity-monitoring \
            --labels=goog-ec-src=vm_add-gcloud \
            --reservation-affinity=any
        '''
        commands.append(cmd.strip())
    
    
    def run_command(command):
        print(f"Running: {command}")
        return subprocess.call(command, shell=True)
    
    
    # #Parallel instance creation
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
        futures = [executor.submit(run_command, cmd) for cmd in commands]
        concurrent.futures.wait(futures)
    
    print("All instances launched.")
    

ERROR: (gcloud.compute.instances.delete) argument INSTANCE_NAMES [INSTANCE_NAMES ...]: Must be specified.
Usage: gcloud compute instances delete INSTANCE_NAMES [INSTANCE_NAMES ...] [optional flags]
  optional flags may be  --delete-disks | --help | --keep-disks | --zone

For detailed information on this command and its flags, run:
  gcloud compute instances delete --help


Running: gcloud compute instances create tsm-sc-000         --project=ucr-ursa-major-lesani-lab         --zone=us-central1-c         --machine-type=e2-highmem-2         --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default         --can-ip-forward         --maintenance-policy=MIGRATE         --provisioning-model=STANDARD         --service-account=961693926925-compute@developer.gserviceaccount.com         --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append         --tags=http-server,https-server         --create-disk=auto-delete=yes,boot=yes,image-family=tsm-sc-family,mode=rw,size=20,type=pd-balanced         --no-shielded-secure-boot         --shielded-vtpm         --shielded-integrity-monitoring         --label

Created [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-006].


NAME        ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP    STATUS
tsm-sc-006  us-central1-c  e2-highmem-2               10.128.0.5   34.28.190.227  RUNNING
Running: gcloud compute instances create tsm-sc-020         --project=ucr-ursa-major-lesani-lab         --zone=us-central1-c         --machine-type=e2-highmem-2         --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default         --can-ip-forward         --maintenance-policy=MIGRATE         --provisioning-model=STANDARD         --service-account=961693926925-compute@developer.gserviceaccount.com         --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append         --tags=http-server,https-server         --create-disk=auto-delete=yes,boo

Created [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-008].


...

NAME        ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP   EXTERNAL_IP    STATUS
tsm-sc-012  us-central1-c  e2-highmem-2               10.128.0.113  34.71.154.146  RUNNING


Created [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-001].


NAME        ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP   EXTERNAL_IP     STATUS
tsm-sc-020  us-central1-c  e2-highmem-2               10.128.0.122  136.114.73.219  RUNNING
NAME        ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP   EXTERNAL_IP    STATUS
tsm-sc-001  us-central1-c  e2-highmem-2               10.128.0.109  34.61.230.149  RUNNING


Created [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-014].


NAME        ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP     STATUS
tsm-sc-014  us-central1-c  e2-highmem-2               10.128.0.6   136.114.107.95  RUNNING
All instances launched.


In [13]:
    # Wait a bit for IPs to propagate
    import time
    time.sleep(30)
    

    # Get IPs
    os.system('gcloud compute instances list --filter="name~\'tsm-sc-\'" '
              '--format="value(networkInterfaces[0].networkIP)" > tsm_ips.txt')
    
    with open('tsm_ips.txt', 'r') as f:
        iplist = [line.strip() for line in f.readlines()]
    
    print("🎯 Instance IPs:", iplist)
    

🎯 Instance IPs: ['10.128.0.118', '10.128.0.109', '10.128.0.119', '10.128.0.15', '10.128.0.114', '10.128.0.112', '10.128.0.5', '10.128.0.120', '10.128.0.108', '10.128.0.121', '10.128.0.106', '10.128.0.115', '10.128.0.113', '10.128.0.117', '10.128.0.6', '10.128.0.116', '10.128.0.107', '10.128.0.110', '10.128.0.14', '10.128.0.111', '10.128.0.122', '10.128.0.123', '10.128.0.124', '10.128.0.125', '10.128.0.126', '10.128.0.127', '10.128.15.192', '10.128.15.193', '10.128.15.194', '10.128.15.195', '10.128.15.196', '10.128.15.197']


In [14]:
    os.system('git add .; git commit -m "testing"; git push')
   

    n_collection = 100
    os.system('make -j8')
    
    

    def kill_stellar_private(i):
        remote_command = f"""\
    cd /home/tejas/stellar-private; \
    sudo pkill stellar-core; \
    """
        
        # Construct the full gcloud command
        command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "{remote_command}"'
        
        print(f"Executing: {command}")
    
        output = os.system(command)
        print(f"Return code for tsm-sc-{i:03}: {output}")
    
    
    results = Parallel(n_jobs=20)(delayed(kill_stellar_private)(i) for i in range(num_nodes))
    
    

    
    
    
    def git_pull_stellar(i):
        command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
    cd stellar-core; \
    git pull"'
        print(command)
        output = os.system(command)
        print(output)
    
    # Execute in parallel like your example
    results = Parallel(n_jobs=20)(delayed(git_pull_stellar)(i) for i in range(len(iplist)))
    print(results)
    

    import shutil
    
    if os.path.exists('../stellar-private'):
        
        shutil.rmtree('../stellar-private')
    os.mkdir('../stellar-private')
    
    
    os.system('cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh')
    
    os.system('cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; ./gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh')
    
    # --- Configuration ---
    line_to_add = "SEND_CUSTOM_MESSAGE=true"
    target_file = "../stellar-private/node1/stellar-core.cfg" 
    
    # --- The os.system() Command ---
    # This command prepends the line to the target_file on your local machine.
    os.system(f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}')
    
    print(f"The line '{line_to_add}' has been prepended to {target_file}.")
    

    def compile_stellar(i):
        command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
    cd stellar-core; \
    make -j16; cd; sudo rm -r stellar-private"'
        print(command)
        output = os.system(command)
        print(output)
    
    # Execute in parallel like your example
    results = Parallel(n_jobs=20)(delayed(compile_stellar)(i) for i in range(len(iplist)))
    print(results)
    

    def clean_stellar_private(i):
        remote_command = f"""\
    cd /home/tejas; \
    sudo rm -r stellar-private; \
    """
        
        # Construct the full gcloud command
        command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "{remote_command}"'
        
        print(f"Executing: {command}")
    
        output = os.system(command)
        print(f"Return code for tsm-sc-{i:03}: {output}")
    
    
    results = Parallel(n_jobs=20)(delayed(clean_stellar_private)(i) for i in range(num_nodes))
    

    def copy_folder_to_instance(i, source_folder = '/home/tejas/stellar-private',destination_path = '/home/tejas/stellar-private' ):
        """
        Constructs and executes the gcloud compute scp command to copy a folder
        to a specific GCP instance.
        """
        instance_name = f"tsm-sc-{i:03}"
        
        # The --recurse flag is crucial for copying folders
        # The format is: gcloud compute scp --recurse [LOCAL_SRC] [USER]@[INSTANCE_NAME]:[REMOTE_DEST]
        command = f'gcloud compute scp --zone "{zone}" --project "{project}" \
    --recurse "{source_folder}" "{instance_name}:{destination_path}"'
    
        print(f"Executing command for {instance_name}: {command}")
        
        # os.system executes the command and returns the exit status (0 for success)
        output = os.system(command)
        
        print(f"Command for {instance_name} finished with exit code: {output}")
        
        return (instance_name, output)
    
    
    results = Parallel(n_jobs=20)(
        delayed(copy_folder_to_instance)(i) for i in range(num_nodes)
    )
    
    

    
    # def setup_stellar_private(i):
    #     command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
    # cd /home/tejas; \
    # cd stellar-private; chmod +x gcp_setup_stellar_private.sh; \
    # ./gcp_setup_stellar_private.sh;"'
    #     print(command)
    #     output = os.system(command)
    #     print(output)
    
    # # Execute in parallel like your example
    # results = Parallel(n_jobs=20)(delayed(setup_stellar_private)(i) for i in range(len(iplist)))
    # print(results)
    
    def run_stellar_private(i):
        # Calculate the node number (assuming i starts at 0, node starts at 1)
        node_number = i + 1 
        instance_name = f"tsm-sc-{i:03}"
        
        # ----------------------------------------------------------------------------------
        # FIX: Use nohup, redirect I/O to a log file, and add ' & disown'
        # '2>&1' redirects stderr to stdout. '> log.txt' redirects stdout to a file.
        # '< /dev/null' ensures the process doesn't wait for input.
        # ----------------------------------------------------------------------------------
        remote_command = f"""\
    cd /home/tejas/stellar-private; \
    nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
    > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
    """
        
        # Construct the full gcloud command
        command = f'gcloud compute ssh --zone "{zone}" "{instance_name}" --project "{project}" --command "{remote_command}"'
        
        print(f"Executing: {command}")
        
        # os.system should now return immediately because the remote shell exits
        output = os.system(command)
        print(f"Return code for {instance_name}: {output}")
    
    # Corrected Loop (to run 0, 1, 2, 3)
    results = Parallel(n_jobs=20)(delayed(run_stellar_private)(i) for i in range(num_nodes))
    # results = Parallel(n_jobs=20)(delayed(run_stellar_private)(i) for i in [3,2,1,0])
    

    # time.sleep(3)
    # for i in range(num_nodes):
    
        # run_stellar_private(num_nodes-i-1)
        # time.sleep(2)
    # run_stellar_private(0)
    print(results)
    print("All SSH commands executed. Nodes should be starting up in the background.")
    
    time.sleep(120)
    

    
    
    def kill_stellar_private(i):
        remote_command = f"""\
    cd /home/tejas/stellar-private; \
    sudo pkill stellar-core; \
    """
        
        # Construct the full gcloud command
        command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "{remote_command}"'
        
        print(f"Executing: {command}")
    
        output = os.system(command)
        print(f"Return code for tsm-sc-{i:03}: {output}")
    
    
    results = Parallel(n_jobs=20)(delayed(kill_stellar_private)(i) for i in range(num_nodes))
    
    

    remote_base_folder = "/home/tejas/stellar-private" # The base path on the GCP instance
    local_base_destination = "/home/tejas/work/experiments/stellar-core/" + "collection_"+str(n_collection)+"_rounds_" + str(num_nodes) + "_test" 
    # local_base_destination = "/home/tejas/work/experiments/stellar-core/" + "collection_" + str(num_nodes) + ""
    
    # Ensure the local base destination directory exists
    os.makedirs(local_base_destination, exist_ok=True)
    
    
    def copy_folder_from_instance(i):
        """
        Constructs and executes the gcloud compute scp command to copy a specific 
        nodeN folder from instance i to a local folder named after the instance.
        """
        instance_name = f"tsm-sc-{i:03}"
        
        # Calculate the node number (assuming i starts at 0, node starts at 1)
        node_number = i + 1 
        node_folder = f"node{node_number}"
    
        # 1. Define the specific REMOTE source path on the instance
        # Example: /home/tejas/stellar-private/node1
        remote_source_path = os.path.join(remote_base_folder, node_folder)
        
        # 2. Define the LOCAL destination path
        # We'll use the instance name for the subfolder to keep backups separate
        local_destination_path = os.path.join(local_base_destination, instance_name)
        os.makedirs(local_destination_path, exist_ok=True)
        
        # The SCp command requires the remote path to be formatted as:
        # [INSTANCE_NAME]:[REMOTE_SRC]
        remote_source = f"{instance_name}:{remote_source_path}"
        
        # The command reverses the source (remote) and destination (local)
        command = f'gcloud compute scp --zone "{zone}" --project "{project}" \
    --recurse "{remote_source}" "{local_destination_path}"'
    
        print(f"Executing command to copy {node_folder} from {instance_name}: {command}")
        
        # os.system executes the command and returns the exit status (0 for success)
        output = os.system(command)
        
        print(f"Copy from {instance_name} finished with exit code: {output}")
        
        return (instance_name, output)
    
    # ---
    # Execute the copy operation in parallel
    # ---
    
    results = Parallel(n_jobs=20)(
        delayed(copy_folder_from_instance)(i) for i in range(3)
    )
    
    print("\n--- Summary of Download Results ---")
    print(results)
    
    
    # # Fetch all existing instance names matching tsm-sc-*
    # fetch_cmd = f'''
    # gcloud compute instances list \
    #     --filter="name~'tsm-sc-'" \
    #     --format="value(name)"
    # '''
    # instances_to_delete = subprocess.check_output(fetch_cmd, shell=True).decode().strip().split('\n')
    # instances_to_delete = [name for name in instances_to_delete if name]  # Remove empty entries
    
    # print("\n➡ Existing instances to delete:", instances_to_delete)
    
    # if instances_to_delete:
    #     # Use parallel deletion
    #     def delete_instance(instance_name):
    #         cmd = f'''
    #         gcloud compute instances delete {instance_name} \
    #             --zone={zone} \
    #             --project={project} \
    #             --quiet
    #         '''
    #         print(f"Deleting: {instance_name}")
    #         return subprocess.call(cmd, shell=True)
    
    #     with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    #         futures = [executor.submit(delete_instance, inst) for inst in instances_to_delete]
    #         concurrent.futures.wait(futures)
    
    #     print("🧹 All old tsm-sc-* instances deleted.\n")
    # else:
    #     print("✔ No previous instances found to delete.\n")
    
    

[main dfb6e5d] testing
 5 files changed, 209 insertions(+), 1187 deletions(-)


To github.com:tejas-shivanand-mane/stellar-core.git
   a29e52b..dfb6e5d  main -> main


make  all-recursive
make[1]: Entering directory '/home/tejas/stellar-core'
Making all in lib
make[2]: Entering directory '/home/tejas/stellar-core/lib'
Making all in ../lib/libsodium
make[3]: Entering directory '/home/tejas/stellar-core/lib/libsodium'
Making all in builds
make[4]: Entering directory '/home/tejas/stellar-core/lib/libsodium/builds'
make[4]: Nothing to be done for 'all'.
make[4]: Leaving directory '/home/tejas/stellar-core/lib/libsodium/builds'
Making all in contrib
make[4]: Entering directory '/home/tejas/stellar-core/lib/libsodium/contrib'
make[4]: Nothing to be done for 'all'.
make[4]: Leaving directory '/home/tejas/stellar-core/lib/libsodium/contrib'
Making all in dist-build
make[4]: Entering directory '/home/tejas/stellar-core/lib/libsodium/dist-build'
make[4]: Nothing to be done for 'all'.
make[4]: Leaving directory '/home/tejas/stellar-core/lib/libsodium/dist-build'
Making all in msvc-scripts
make[4]: Entering directory '/home/tejas/stellar-core/lib/libsodium/msvc-

From https://github.com/tejas-shivanand-mane/stellar-core
   01f9b6a..dfb6e5d  main       -> origin/main
From https://github.com/tejas-shivanand-mane/stellar-core
   01f9b6a..dfb6e5d  main       -> origin/main
From https://github.com/tejas-shivanand-mane/stellar-core
   01f9b6a..dfb6e5d  main       -> origin/main
From https://github.com/tejas-shivanand-mane/stellar-core
   01f9b6a..dfb6e5d  main       -> origin/main
From https://github.com/tejas-shivanand-mane/stellar-core
   01f9b6a..dfb6e5d  main       -> origin/main
From https://github.com/tejas-shivanand-mane/stellar-core
   01f9b6a..dfb6e5d  main       -> origin/main


Updating 01f9b6a..dfb6e5d
Fast-forward
 .ipynb_checkpoints/SCPPost-checkpoint.ipynb |  188 +++
 SCPPost.ipynb                               |  176 +++
 SetupGCP.ipynb                              | 2170 ++++-----------------------
 latency.png                                 |  Bin 88937 -> 65383 bytes
 post.ipynb                                  |  418 ++----
 src/overlay/OverlayManagerImpl.cpp          |  138 +-
 throughput.png                              |  Bin 106744 -> 76407 bytes
 tsm_ips.txt                                 |   36 +-
 8 files changed, 903 insertions(+), 2223 deletions(-)
 create mode 100644 .ipynb_checkpoints/SCPPost-checkpoint.ipynb
 create mode 100644 SCPPost.ipynb
Updating 01f9b6a..dfb6e5d
Fast-forward
Updating 01f9b6a..dfb6e5d
Fast-forward
Updating 01f9b6a..dfb6e5d
Fast-forward
 .ipynb_checkpoints/SCPPost-checkpoint.ipynb |  188 +++
 SCPPost.ipynb                               |  176 +++
 SetupGCP.ipynb                              | 2170 ++++---------------

...

/bin/bash ../libtool  --tag=CXX   --mode=link g++ -std=c++17  -g -O2 -fno-omit-frame-pointer  -pthread -DFMT_HEADER_ONLY=1 -Wall -Wno-unused-command-line-argument -Wno-unused-local-typedef -Wno-unknown-warning-option -Werror=unused-result   -o stellar-core main/StellarCoreVersion.o main/XDRFilesSha256.o bucket/BucketApplicator.o bucket/BucketBase.o bucket/BucketIndexUtils.o bucket/BucketInputIterator.o bucket/BucketListBase.o bucket/BucketListSnapshotBase.o bucket/BucketManager.o bucket/BucketMergeMap.o bucket/BucketOutputIterator.o bucket/BucketSnapshot.o bucket/BucketSnapshotManager.o bucket/BucketUtils.o bucket/DiskIndex.o bucket/FutureBucket.o bucket/HotArchiveBucket.o bucket/HotArchiveBucketIndex.o bucket/HotArchiveBucketList.o bucket/InMemoryIndex.o bucket/LiveBucket.o bucket/LiveBucketIndex.o bucket/LiveBucketList.o bucket/MergeKey.o bucket/SearchableBucketList.o catchup/ApplyBucketsWork.o catchup/ApplyBufferedLedgersWork.o catchup/ApplyCheckpointWork.o catchup/ApplyLedgerWork.o

At global scope:
cc1plus: note: unrecognized command-line option ‘-Wno-unknown-warning-option’ may have been intended to silence earlier diagnostics
cc1plus: note: unrecognized command-line option ‘-Wno-unused-local-typedef’ may have been intended to silence earlier diagnostics
cc1plus: note: unrecognized command-line option ‘-Wno-unused-command-line-argument’ may have been intended to silence earlier diagnostics


/bin/bash ../libtool  --tag=CXX   --mode=link g++ -std=c++17  -g -O2 -fno-omit-frame-pointer  -pthread -DFMT_HEADER_ONLY=1 -Wall -Wno-unused-command-line-argument -Wno-unused-local-typedef -Wno-unknown-warning-option -Werror=unused-result   -o stellar-core main/StellarCoreVersion.o main/XDRFilesSha256.o bucket/BucketApplicator.o bucket/BucketBase.o bucket/BucketIndexUtils.o bucket/BucketInputIterator.o bucket/BucketListBase.o bucket/BucketListSnapshotBase.o bucket/BucketManager.o bucket/BucketMergeMap.o bucket/BucketOutputIterator.o bucket/BucketSnapshot.o bucket/BucketSnapshotManager.o bucket/BucketUtils.o bucket/DiskIndex.o bucket/FutureBucket.o bucket/HotArchiveBucket.o bucket/HotArchiveBucketIndex.o bucket/HotArchiveBucketList.o bucket/InMemoryIndex.o bucket/LiveBucket.o bucket/LiveBucketIndex.o bucket/LiveBucketList.o bucket/MergeKey.o bucket/SearchableBucketList.o catchup/ApplyBucketsWork.o catchup/ApplyBufferedLedgersWork.o catchup/ApplyCheckpointWork.o catchup/ApplyLedgerWork.o

rm: cannot remove 'stellar-private': No such file or directory
rm: cannot remove 'stellar-private': No such file or directory
rm: cannot remove 'stellar-private': No such file or directory
rm: cannot remove 'stellar-private': No such file or directory
rm: cannot remove 'stellar-private': No such file or directory
rm: cannot remove 'stellar-private': No such file or directory
rm: cannot remove 'stellar-private': No such file or directory
rm: cannot remove 'stellar-private': No such file or directory
rm: cannot remove 'stellar-private': No such file or directory
rm: cannot remove 'stellar-private': No such file or directory
rm: cannot remove 'stellar-private': No such file or directory
rm: cannot remove 'stellar-private': No such file or directory
rm: cannot remove 'stellar-private': No such file or directory
rm: cannot remove 'stellar-private': No such file or directory
rm: cannot remove 'stellar-private': No such file or directory
rm: cannot remove 'stellar-private': No such file or di

[None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]
All SSH commands executed. Nodes should be starting up in the background.

--- Summary of Download Results ---
[('tsm-sc-000', 0), ('tsm-sc-001', 0), ('tsm-sc-002', 0)]


In [15]:

    # # Fetch all existing instance names matching tsm-sc-*
    # fetch_cmd = f'''
    # gcloud compute instances list \
    #     --filter="name~'tsm-sc-'" \
    #     --format="value(name)"
    # '''
    # instances_to_delete = subprocess.check_output(fetch_cmd, shell=True).decode().strip().split('\n')
    # instances_to_delete = [name for name in instances_to_delete if name]  # Remove empty entries
    
    # print("\n➡ Existing instances to delete:", instances_to_delete)
    
    # if instances_to_delete:
    #     # Use parallel deletion
    #     def delete_instance(instance_name):
    #         cmd = f'''
    #         gcloud compute instances delete {instance_name} \
    #             --zone={zone} \
    #             --project={project} \
    #             --quiet
    #         '''
    #         print(f"Deleting: {instance_name}")
    #         return subprocess.call(cmd, shell=True)
    
    #     with concurrent.futures.ThreadPoolExecutor(max_workers=50) as executor:
    #         futures = [executor.submit(delete_instance, inst) for inst in instances_to_delete]
    #         concurrent.futures.wait(futures)
    
    #     print("🧹 All old tsm-sc-* instances deleted.\n")
    # else:
    #     print("✔ No previous instances found to delete.\n")

Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-006" --project "ucr-ursa-major-lesani-lab" --command "cd /home/tejas/stellar-private; sudo pkill stellar-core; "
Return code for tsm-sc-006: 256
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-027" --project "ucr-ursa-major-lesani-lab" --command "cd /home/tejas/stellar-private; sudo pkill stellar-core; "
Return code for tsm-sc-027: 256
gcloud compute ssh --zone "us-central1-c" "tsm-sc-019" --project "ucr-ursa-major-lesani-lab" --command "cd stellar-core; git pull"
0
gcloud compute ssh --zone "us-central1-c" "tsm-sc-006" --project "ucr-ursa-major-lesani-lab" --command "cd stellar-core; make -j16; cd; sudo rm -r stellar-private"
0
gcloud compute ssh --zone "us-central1-c" "tsm-sc-020" --project "ucr-ursa-major-lesani-lab" --command "cd stellar-core; make -j16; cd; sudo rm -r stellar-private"
0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-015" --project "ucr-ursa-major-lesani-lab" --command "cd /home

Exception ignored in: <function ResourceTracker.__del__ at 0x75c49c182020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x7f5f06d82020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ 

Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-003" --project "ucr-ursa-major-lesani-lab" --command "cd /home/tejas/stellar-private; sudo pkill stellar-core; "
Return code for tsm-sc-003: 256
gcloud compute ssh --zone "us-central1-c" "tsm-sc-002" --project "ucr-ursa-major-lesani-lab" --command "cd stellar-core; git pull"
0
gcloud compute ssh --zone "us-central1-c" "tsm-sc-021" --project "ucr-ursa-major-lesani-lab" --command "cd stellar-core; git pull"
0
gcloud compute ssh --zone "us-central1-c" "tsm-sc-011" --project "ucr-ursa-major-lesani-lab" --command "cd stellar-core; make -j16; cd; sudo rm -r stellar-private"
0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-002" --project "ucr-ursa-major-lesani-lab" --command "cd /home/tejas; sudo rm -r stellar-private; "
Return code for tsm-sc-002: 256
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-025" --project "ucr-ursa-major-lesani-lab" --command "cd /home/tejas; sudo rm -r stellar-private; "
Return c

Exception ignored in: <function ResourceTracker.__del__ at 0x78afa897e020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-005" --project "ucr-ursa-major-lesani-lab" --command "cd /home/tejas/stellar-private; sudo pkill stellar-core; "
Return code for tsm-sc-005: 256
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-025" --project "ucr-ursa-major-lesani-lab" --command "cd /home/tejas/stellar-private; sudo pkill stellar-core; "
Return code for tsm-sc-025: 256
gcloud compute ssh --zone "us-central1-c" "tsm-sc-015" --project "ucr-ursa-major-lesani-lab" --command "cd stellar-core; git pull"
0
gcloud compute ssh --zone "us-central1-c" "tsm-sc-028" --project "ucr-ursa-major-lesani-lab" --command "cd stellar-core; git pull"
0
gcloud compute ssh --zone "us-central1-c" "tsm-sc-015" --project "ucr-ursa-major-lesani-lab" --command "cd stellar-core; make -j16; cd; sudo rm -r stellar-private"
0
gcloud compute ssh --zone "us-central1-c" "tsm-sc-027" --project "ucr-ursa-major-lesani-lab" --command "cd stellar-core; make -j16; cd; sudo rm -r stellar-p

...

Exception ignored in: <function ResourceTracker.__del__ at 0x79ce31982020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-014" --project "ucr-ursa-major-lesani-lab" --command "cd /home/tejas/stellar-private; sudo pkill stellar-core; "
Return code for tsm-sc-014: 256
gcloud compute ssh --zone "us-central1-c" "tsm-sc-004" --project "ucr-ursa-major-lesani-lab" --command "cd stellar-core; git pull"
0
gcloud compute ssh --zone "us-central1-c" "tsm-sc-022" --project "ucr-ursa-major-lesani-lab" --command "cd stellar-core; git pull"
0
gcloud compute ssh --zone "us-central1-c" "tsm-sc-009" --project "ucr-ursa-major-lesani-lab" --command "cd stellar-core; make -j16; cd; sudo rm -r stellar-private"
0
gcloud compute ssh --zone "us-central1-c" "tsm-sc-026" --project "ucr-ursa-major-lesani-lab" --command "cd stellar-core; make -j16; cd; sudo rm -r stellar-private"
0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-010" --project "ucr-ursa-major-lesani-lab" --command "cd /home/tejas; sudo rm -r stellar-private; "
Return code for tsm-sc-010: 256
Exe

Exception ignored in: <function ResourceTracker.__del__ at 0x714608f86020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x75f647386020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-012" --project "ucr-ursa-major-lesani-lab" --command "cd /home/tejas/stellar-private; sudo pkill stellar-core; "
Return code for tsm-sc-012: 256
gcloud compute ssh --zone "us-central1-c" "tsm-sc-006" --project "ucr-ursa-major-lesani-lab" --command "cd stellar-core; git pull"
0
gcloud compute ssh --zone "us-central1-c" "tsm-sc-007" --project "ucr-ursa-major-lesani-lab" --command "cd stellar-core; make -j16; cd; sudo rm -r stellar-private"
0
gcloud compute ssh --zone "us-central1-c" "tsm-sc-022" --project "ucr-ursa-major-lesani-lab" --command "cd stellar-core; make -j16; cd; sudo rm -r stellar-private"
0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-011" --project "ucr-ursa-major-lesani-lab" --command "cd /home/tejas; sudo rm -r stellar-private; "
Return code for tsm-sc-011: 256
Executing command for tsm-sc-004: gcloud compute scp --zone "us-central1-c" --project "ucr-ursa-major-lesani-lab" --recurse "/home/tejas

Exception ignored in: <function ResourceTracker.__del__ at 0x75937bb8e020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
